# Mediador de retroalimentación entre pares — piloto
**Flujo:** estudiante retroalimentador → IA (bucle de hasta *N* rondas) → estudiante destinatario, con registro de cada iteración para el profesor.

**Componentes**
- **LangGraph**: grafo `evaluar → esperar (interrupt) → decidir → evaluar | rechazar | entregar`, con estado persistido en SQLite. Cada pausa del bucle se reanuda con lo que el estudiante escribe.
- **Bloqueo de contenido inapropiado**: la IA marca `bloqueado=true` ante insultos, ataques personales o texto que no es retroalimentación. Un comentario bloqueado **no puede enviarse** aunque el estudiante escriba ENVIAR: pasa al nodo `rechazar`, se le explica y debe reescribir. Toda versión nueva (incluida la última) pasa por la IA antes de enviarse. Los intentos bloqueados quedan en el Excel y, si configuras `PROFESOR_CHAT_ID`, el profesor recibe un aviso por Telegram (`/miid` te da tu id).
- **Modelo**: `gpt-5.4-mini` con salida estructurada (puntajes 1–4 en cinco criterios + sugerencias).
- **Prompt** en archivo aparte (`prompt_mediador.md`) para poder iterarlo sin tocar el código.
- **Registro** `registro_retroalimentacion.xlsx`: una fila por borrador evaluado (texto, puntajes, bloqueo y motivo, sugerencias de la IA, acciones del estudiante, versión final).
- **Canal**: bot de Telegram para el bucle; entrega final por Telegram y/o correo.

**Antes de empezar** — en Colab, panel *Secretos* (🔑): `OPENAI_API_KEY`, `TELEGRAM_BOT_TOKEN` (créalo con @BotFather). Opcionales: `GMAIL_USER`, `GMAIL_APP_PASSWORD` (entrega por correo), `PROFESOR_CHAT_ID` (avisos de bloqueo).

> ⚠️ Los archivos `.xlsx`/`.sqlite` viven en la sesión de Colab: descárgalos al final (última celda) o monta Drive y cambia las rutas en la configuración.


In [ ]:
%pip install -q -U langgraph langgraph-checkpoint-sqlite langchain-openai python-telegram-bot openpyxl pandas nest_asyncio

## 1. Prompt del mediador (archivo aparte)
Edita este archivo para ajustar criterios, tono o reglas. El texto entre `{}` se rellena en cada ronda.

In [ ]:
%%writefile prompt_mediador.md
Eres un **mediador pedagógico de retroalimentación entre pares** en un curso universitario.

Un estudiante (el *retroalimentador*) ha escrito un comentario de retroalimentación sobre el trabajo de un compañero (el *destinatario*). Tu tarea NO es corregir el trabajo del compañero ni reescribir el comentario: es ayudar al retroalimentador a que su comentario sea más útil para quien lo recibe, manteniendo su propia voz.

## Contexto de la tarea evaluada
{tarea}

## Criterios de calidad de la retroalimentación (escala 1–4)

1. **Especificidad** — ¿Señala pasajes, secciones o decisiones concretas del trabajo, o habla en general ("está bien", "le falta")?
   1 = totalmente genérico · 4 = apunta a elementos concretos e identificables.
2. **Tono** — ¿Es respetuoso y se dirige al trabajo y no a la persona? ¿Evita sarcasmo, juicios globales o condescendencia?
   1 = hiriente o despectivo · 4 = respetuoso, constructivo y directo.
3. **Accionabilidad** — ¿El destinatario sabría qué hacer después de leerlo? ¿Hay sugerencias o preguntas que abran un camino de mejora?
   1 = no indica ningún paso posible · 4 = propone acciones o preguntas claras.
4. **Equilibrio** — ¿Reconoce fortalezas reales además de señalar problemas? (No se trata de "endulzar", sino de que el destinatario sepa qué conservar.)
   1 = solo críticas o solo elogios vacíos · 4 = fortalezas y mejoras, ambas fundamentadas.
5. **Alineación** — ¿Se relaciona con los objetivos o criterios de la tarea, o se desvía hacia aspectos irrelevantes (formato, gustos personales)?
   1 = desconectado de la tarea · 4 = anclado en los criterios de la tarea.

## Cómo debes actuar

- Evalúa el borrador con honestidad; no infles los puntajes.
- Escribe las **fortalezas** del comentario en una o dos frases (qué está haciendo bien el retroalimentador).
- Propón como máximo **tres sugerencias** concretas y breves. Prefiere **preguntas** que ayuden al retroalimentador a completar su idea ("¿En qué parte del informe notaste eso?", "¿Qué cambio concreto le propondrías?") en lugar de darle la frase ya escrita.
- **Nunca** entregues una versión reescrita completa del comentario. El texto debe seguir siendo del estudiante.
- Marca `listo_para_enviar = true` cuando todos los criterios estén en 3 o más y no haya problemas de tono. En ese caso dilo de forma breve y positiva: no obligues a iterar por iterar.
- Cuando `supera_rondas_sugeridas` sea verdadero, ya no propongas más mejoras de fondo: limítate a verificar el tono y, si no hay bloqueo, marca `listo_para_enviar = true` con un mensaje muy breve.

## Bloqueo (el sistema NO permitirá enviar el comentario)

Marca `bloqueado = true` únicamente si el texto contiene alguno de estos elementos:
- insultos, groserías o burlas dirigidas al compañero;
- ataques o descalificaciones a la persona (inteligencia, esfuerzo, origen, apariencia, etc.) en lugar de al trabajo;
- contenido discriminatorio, amenazante o de acoso;
- un mensaje que no es retroalimentación (por ejemplo, un insulto dirigido a la IA, spam o texto sin relación con la tarea).

Un comentario duro, seco o poco amable **no** es bloqueo: eso se refleja con un puntaje bajo en Tono y una sugerencia. El bloqueo es para lo que no debe llegar al compañero bajo ninguna forma.

Cuando bloquees, escribe en `motivo_bloqueo` una frase objetiva (sin repetir el insulto) y en `mensaje_al_retroalimentador` explica con calma, sin sermonear, que el comentario no puede enviarse así y qué tendría que cambiar. Pon los cinco puntajes en 1 si el texto no es retroalimentación.
- Redacta `mensaje_al_retroalimentador` en español, en segunda persona, tono cercano y profesional, máximo ~120 palabras. Es el texto que verá el estudiante en el chat. No incluyas los puntajes numéricos en ese mensaje (se muestran aparte).

## Información de esta ronda
- Ronda: {iteracion} (rondas sugeridas: {max_iteraciones})
- ¿Supera las rondas sugeridas?: {supera_rondas_sugeridas}

## Borrador del retroalimentador
"""
{borrador}
"""


## 2. Directorio de estudiantes
`alias` es lo que escriben en Telegram (`/soy ana`, `/retro beto`). El correo solo se usa si configuras Gmail.

In [ ]:
%%writefile directorio.csv
alias,nombre,correo
ana,Ana Pérez,ana.perez@mail.pucv.cl
beto,Roberto Díaz,roberto.diaz@mail.pucv.cl
carla,Carla Soto,carla.soto@mail.pucv.cl
dani,Daniela Rojas,daniela.rojas@mail.pucv.cl


## 3. Configuración

In [ ]:
# @title Configuración
import os, json, uuid, datetime as dt
from pathlib import Path

# --- Claves: en Colab, agrégalas en el panel "Secretos" (ícono de llave) ---
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"]     = userdata.get("OPENAI_API_KEY")
    os.environ["TELEGRAM_BOT_TOKEN"] = userdata.get("TELEGRAM_BOT_TOKEN")
    # Opcionales, solo si quieres entregar por correo:
    for k in ("GMAIL_USER", "GMAIL_APP_PASSWORD", "PROFESOR_CHAT_ID"):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
except ImportError:
    pass  # fuera de Colab: exporta las variables de entorno antes de abrir el notebook

assert os.environ.get("OPENAI_API_KEY"), "Falta OPENAI_API_KEY"

MODELO            = "gpt-5.4-mini"
MAX_ITERACIONES   = 3                         # rondas "sugeridas": después la IA solo verifica tono/bloqueo
ARCHIVO_PROMPT    = Path("prompt_mediador.md")
ARCHIVO_DIRECTORIO= Path("directorio.csv")
ARCHIVO_XLSX      = Path("registro_retroalimentacion.xlsx")
ARCHIVO_DB        = Path("checkpoints.sqlite")    # estado del grafo (LangGraph)
ARCHIVO_CHATS     = Path("chats_telegram.json")   # alias -> chat_id de Telegram

# Descripción de la tarea que los estudiantes están retroalimentando.
# Puede cambiarse desde Telegram con /tarea <texto>.
TAREA = (
    "Informe de laboratorio N°2. Criterios: (1) hipótesis clara y justificada, "
    "(2) procedimiento reproducible, (3) análisis de resultados con evidencia, "
    "(4) conclusiones coherentes con los datos."
)

print("Modelo:", MODELO, "| Máx. iteraciones:", MAX_ITERACIONES)


## 4. Modelo evaluador

In [ ]:
# @title Modelo evaluador (salida estructurada)
from typing import List
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

class Evaluacion(BaseModel):
    """Evaluación de la IA sobre el borrador de retroalimentación."""
    especificidad: int = Field(ge=1, le=4, description="1-4 según el criterio Especificidad")
    tono:          int = Field(ge=1, le=4, description="1-4 según el criterio Tono")
    accionabilidad:int = Field(ge=1, le=4, description="1-4 según el criterio Accionabilidad")
    equilibrio:    int = Field(ge=1, le=4, description="1-4 según el criterio Equilibrio")
    alineacion:    int = Field(ge=1, le=4, description="1-4 según el criterio Alineación")
    fortalezas:    str = Field(description="Qué hace bien el retroalimentador (1-2 frases)")
    sugerencias:   List[str] = Field(description="Máximo 3 sugerencias o preguntas concretas", max_length=3)
    listo_para_enviar: bool = Field(description="True si el comentario ya es adecuado para enviarse")
    bloqueado: bool = Field(description="True si contiene insultos, ataques personales, contenido discriminatorio o no es retroalimentación. El sistema impedirá el envío.")
    motivo_bloqueo: str = Field(default="", description="Razón objetiva del bloqueo (vacío si no está bloqueado)")
    mensaje_al_retroalimentador: str = Field(description="Texto que verá el estudiante (máx ~120 palabras)")

    @property
    def puntaje_global(self) -> float:
        return round((self.especificidad + self.tono + self.accionabilidad
                      + self.equilibrio + self.alineacion) / 5, 2)

PLANTILLA_PROMPT = ARCHIVO_PROMPT.read_text(encoding="utf-8")

llm = ChatOpenAI(model=MODELO)
evaluador = llm.with_structured_output(Evaluacion)

def evaluar_borrador(borrador: str, tarea: str, iteracion: int, max_iteraciones: int) -> Evaluacion:
    prompt = PLANTILLA_PROMPT.format(
        tarea=tarea, borrador=borrador, iteracion=iteracion,
        max_iteraciones=max_iteraciones,
        supera_rondas_sugeridas="sí" if iteracion > max_iteraciones else "no",
    )
    return evaluador.invoke(prompt)

MENSAJE_RECHAZO = (
    "🚫 Este comentario no se puede enviar tal como está.\n\n{motivo}\n\n"
    "Escribe una nueva versión centrada en el trabajo de tu compañero, o usa /cancelar."
)

def formatear_evaluacion(ev: Evaluacion, iteracion: int, max_iteraciones: int) -> str:
    """Mensaje que se muestra al retroalimentador (Telegram / consola)."""
    if ev.bloqueado:
        return (f"🚫 Ronda {iteracion}\n\n{ev.mensaje_al_retroalimentador.strip()}\n\n"
                "Este comentario no podrá enviarse hasta que lo reformules. "
                "Escribe una nueva versión o usa /cancelar.")
    barras = lambda n: "●" * n + "○" * (4 - n)
    lineas = [
        f"🔎 Ronda {iteracion}" + (f" de {max_iteraciones}" if iteracion <= max_iteraciones else ""),
        "",
        f"Especificidad  {barras(ev.especificidad)}",
        f"Tono           {barras(ev.tono)}",
        f"Accionabilidad {barras(ev.accionabilidad)}",
        f"Equilibrio     {barras(ev.equilibrio)}",
        f"Alineación     {barras(ev.alineacion)}",
        "",
        ev.mensaje_al_retroalimentador.strip(),
    ]
    if ev.sugerencias:
        lineas += [""] + [f"• {s}" for s in ev.sugerencias]
    if ev.listo_para_enviar:
        pie = "✅ Tu comentario está listo. Escribe ENVIAR, o pega una nueva versión si quieres ajustarlo."
    elif iteracion < max_iteraciones:
        pie = "✏️ Pega una nueva versión para seguir mejorándolo, o escribe ENVIAR si prefieres mandarlo así."
    else:
        pie = "⚠️ Ya usaste las rondas sugeridas. Escribe ENVIAR o pega una última versión (toda versión se revisa antes de enviarse)."
    return "\n".join(lineas + ["", pie])


## 5. Registro en Excel

In [ ]:
# @title Registro en Excel (para el profesor)
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

COLUMNAS = [
    "thread_id", "fecha_hora", "tarea", "retroalimentador", "destino", "iteracion",
    "texto", "especificidad", "tono", "accionabilidad", "equilibrio", "alineacion",
    "puntaje_global", "listo_para_enviar", "bloqueado", "motivo_bloqueo",
    "fortalezas", "sugerencias_IA", "accion_retroalimentador", "es_version_final",
]
ANCHOS = {"texto": 60, "motivo_bloqueo": 35, "fortalezas": 40, "sugerencias_IA": 50, "tarea": 30, "fecha_hora": 19}

def _abrir():
    if ARCHIVO_XLSX.exists():
        wb = load_workbook(ARCHIVO_XLSX)
        return wb, wb["iteraciones"]
    wb = Workbook()
    ws = wb.active
    ws.title = "iteraciones"
    ws.append(COLUMNAS)
    for i, col in enumerate(COLUMNAS, start=1):
        c = ws.cell(row=1, column=i)
        c.font = Font(bold=True, color="FFFFFF")
        c.fill = PatternFill("solid", fgColor="2F5496")
        ws.column_dimensions[get_column_letter(i)].width = ANCHOS.get(col, 16)
    ws.freeze_panes = "A2"
    return wb, ws

def registrar_iteracion(thread_id, tarea, retroalimentador, destino, iteracion, texto, ev):
    """Una fila por cada borrador evaluado por la IA."""
    wb, ws = _abrir()
    ws.append([
        thread_id, dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S"), tarea,
        retroalimentador, destino, iteracion, texto,
        ev.especificidad, ev.tono, ev.accionabilidad, ev.equilibrio, ev.alineacion,
        ev.puntaje_global, ev.listo_para_enviar, ev.bloqueado, ev.motivo_bloqueo,
        ev.fortalezas, "\n".join(f"- {s}" for s in ev.sugerencias),
        "", False,
    ])
    for c in ws[ws.max_row]:
        c.alignment = Alignment(wrap_text=True, vertical="top")
    wb.save(ARCHIVO_XLSX)

def _fila(ws, thread_id, iteracion):
    ci, cj = COLUMNAS.index("thread_id") + 1, COLUMNAS.index("iteracion") + 1
    for r in range(ws.max_row, 1, -1):
        if ws.cell(r, ci).value == thread_id and ws.cell(r, cj).value == iteracion:
            return r
    return None

def actualizar_accion(thread_id, iteracion, accion):
    """Qué hizo el retroalimentador tras ver la evaluación: ENVIAR o REESCRIBIR."""
    wb, ws = _abrir()
    r = _fila(ws, thread_id, iteracion)
    if r:
        c = ws.cell(r, COLUMNAS.index("accion_retroalimentador") + 1)
        c.value = f"{c.value} → {accion}" if c.value else accion
        wb.save(ARCHIVO_XLSX)

def registrar_final(thread_id, iteracion):
    """Marca la versión que efectivamente se envió al destinatario (siempre fue evaluada)."""
    wb, ws = _abrir()
    r = _fila(ws, thread_id, iteracion)
    if r:
        ws.cell(r, COLUMNAS.index("es_version_final") + 1).value = True
        wb.save(ARCHIVO_XLSX)

print("Registro en:", ARCHIVO_XLSX.resolve())


## 6. Grafo LangGraph

In [ ]:
# @title Grafo LangGraph (bucle retroalimentador ↔ IA, con bloqueo de contenido ofensivo)
import sqlite3, operator
from typing import TypedDict, Annotated, List, Dict, Any, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_core.runnables import RunnableConfig

class Estado(TypedDict, total=False):
    retroalimentador: str
    destino: str
    tarea: str
    borrador: str                 # versión actual del comentario (siempre evaluada antes de enviarse)
    iteracion: int                # ronda actual
    max_iteraciones: int          # rondas "sugeridas": después, la IA solo verifica tono
    evaluacion: Dict[str, Any]    # última evaluación de la IA
    bloqueado: bool               # la versión actual NO puede enviarse
    intentos_bloqueados: int      # veces que intentó ENVIAR un texto bloqueado
    evento: Literal["evaluacion", "rechazo"]   # qué se le muestra al estudiante en la pausa
    respuesta: Literal["ENVIAR", "REESCRIBIR"]
    historial: Annotated[List[Dict[str, Any]], operator.add]
    mensaje_final: str

def _tid(config: RunnableConfig) -> str:
    return config["configurable"]["thread_id"]

# ---------- nodos ----------
def nodo_evaluar(state: Estado, config: RunnableConfig) -> Estado:
    ev = evaluar_borrador(state["borrador"], state["tarea"],
                          state["iteracion"], state["max_iteraciones"])
    registrar_iteracion(_tid(config), state["tarea"], state["retroalimentador"],
                        state["destino"], state["iteracion"], state["borrador"], ev)
    return {
        "evaluacion": ev.model_dump() | {"puntaje_global": ev.puntaje_global},
        "bloqueado": ev.bloqueado,
        "evento": "evaluacion",
        "historial": [{"iteracion": state["iteracion"], "texto": state["borrador"],
                       "puntaje": ev.puntaje_global, "bloqueado": ev.bloqueado}],
    }

def nodo_esperar(state: Estado, config: RunnableConfig) -> Estado:
    # Pausa el grafo hasta que el retroalimentador responda (Telegram / consola).
    respuesta = interrupt({
        "evento": state["evento"],
        "evaluacion": state["evaluacion"],
        "iteracion": state["iteracion"],
        "max_iteraciones": state["max_iteraciones"],
    })
    texto = (respuesta or "").strip()
    if texto.upper() in ("ENVIAR", "/ENVIAR"):
        return {"respuesta": "ENVIAR"}
    actualizar_accion(_tid(config), state["iteracion"], "REESCRIBIR")
    return {"respuesta": "REESCRIBIR", "borrador": texto, "iteracion": state["iteracion"] + 1}

def decidir(state: Estado) -> str:
    if state["respuesta"] == "REESCRIBIR":
        return "evaluar"                       # toda versión nueva pasa por la IA
    return "rechazar" if state.get("bloqueado") else "entregar"

def nodo_rechazar(state: Estado, config: RunnableConfig) -> Estado:
    # El estudiante quiso ENVIAR un texto bloqueado: no se entrega, se le explica y vuelve a esperar.
    actualizar_accion(_tid(config), state["iteracion"], "INTENTO_ENVIO_BLOQUEADO")
    return {"evento": "rechazo", "intentos_bloqueados": state.get("intentos_bloqueados", 0) + 1}

def nodo_entregar(state: Estado, config: RunnableConfig) -> Estado:
    actualizar_accion(_tid(config), state["iteracion"], "ENVIAR")
    registrar_final(_tid(config), state["iteracion"])
    mensaje = (f"Retroalimentación de {state['retroalimentador']} sobre tu trabajo "
               f"({state['tarea'][:60]}...):\n\n{state['borrador']}")
    return {"mensaje_final": mensaje}

# ---------- construcción ----------
def construir_grafo(checkpointer):
    g = StateGraph(Estado)
    g.add_node("evaluar", nodo_evaluar)
    g.add_node("esperar", nodo_esperar)
    g.add_node("rechazar", nodo_rechazar)
    g.add_node("entregar", nodo_entregar)
    g.add_edge(START, "evaluar")
    g.add_edge("evaluar", "esperar")
    g.add_conditional_edges("esperar", decidir,
                            {"evaluar": "evaluar", "rechazar": "rechazar", "entregar": "entregar"})
    g.add_edge("rechazar", "esperar")
    g.add_edge("entregar", END)
    return g.compile(checkpointer=checkpointer)

conn = sqlite3.connect(ARCHIVO_DB, check_same_thread=False)
grafo = construir_grafo(SqliteSaver(conn))

# ---------- API sencilla usada por consola y Telegram ----------
def iniciar_sesion(thread_id, retroalimentador, destino, borrador, tarea=None):
    """Arranca el bucle. Devuelve (terminado, texto_para_mostrar, bloqueado)."""
    estado = {"retroalimentador": retroalimentador, "destino": destino,
              "tarea": tarea or TAREA, "borrador": borrador, "iteracion": 1,
              "max_iteraciones": MAX_ITERACIONES, "intentos_bloqueados": 0, "historial": []}
    return _interpretar(grafo.invoke(estado, {"configurable": {"thread_id": thread_id}}))

def continuar_sesion(thread_id, respuesta_estudiante):
    """Reanuda el bucle con lo que escribió el retroalimentador."""
    return _interpretar(grafo.invoke(Command(resume=respuesta_estudiante),
                                     {"configurable": {"thread_id": thread_id}}))

def _interpretar(resultado):
    if "__interrupt__" in resultado:
        p = resultado["__interrupt__"][0].value
        ev = Evaluacion(**{k: v for k, v in p["evaluacion"].items() if k != "puntaje_global"})
        if p["evento"] == "rechazo":
            return False, MENSAJE_RECHAZO.format(motivo=ev.motivo_bloqueo or ev.mensaje_al_retroalimentador), True
        return False, formatear_evaluacion(ev, p["iteracion"], p["max_iteraciones"]), ev.bloqueado
    return True, resultado["mensaje_final"], False

try:
    from IPython.display import Image, display
    display(Image(grafo.get_graph().draw_mermaid_png()))
except Exception:
    print(grafo.get_graph().draw_mermaid())


## 7. Prueba en consola (opcional)
Útil para calibrar el prompt antes de abrir el bot a los estudiantes.

In [ ]:
# @title Prueba rápida en consola (sin Telegram)
# Simula a un retroalimentador: escribe ENVIAR o pega una nueva versión cuando se te pida.
tid = f"consola-{uuid.uuid4().hex[:8]}"
borrador = input("Pega el borrador de retroalimentación:\n> ")
terminado, salida, _ = iniciar_sesion(tid, retroalimentador="prueba", destino="compañero", borrador=borrador)
while not terminado:
    print("\n" + salida + "\n")
    terminado, salida, _ = continuar_sesion(tid, input("> "))
print("\n📨 Mensaje entregado al destinatario:\n" + salida)
print("\nRevisa el archivo:", ARCHIVO_XLSX)


## 8. Entrega y bot de Telegram
La celda del bot queda corriendo mientras dure el piloto; detenla (■) para apagarlo. El estado del grafo está en SQLite, así que si reinicias la celda las sesiones en curso se pueden retomar con el mismo `thread_id`.

In [ ]:
# @title Directorio de estudiantes y entrega (Telegram y/o correo)
import csv, smtplib
from email.message import EmailMessage

def cargar_directorio():
    with open(ARCHIVO_DIRECTORIO, encoding="utf-8") as f:
        return {r["alias"].strip().lower(): r for r in csv.DictReader(f)}

DIRECTORIO = cargar_directorio()
print("Estudiantes en el directorio:", ", ".join(DIRECTORIO))

def cargar_chats():
    return json.loads(ARCHIVO_CHATS.read_text()) if ARCHIVO_CHATS.exists() else {}

def guardar_chat(alias, chat_id):
    chats = cargar_chats()
    chats[alias] = chat_id
    ARCHIVO_CHATS.write_text(json.dumps(chats, indent=2))

def enviar_correo(destinatario, asunto, cuerpo) -> bool:
    """Envía por Gmail SMTP si hay GMAIL_USER y GMAIL_APP_PASSWORD; si no, se omite."""
    user, pwd = os.environ.get("GMAIL_USER"), os.environ.get("GMAIL_APP_PASSWORD")
    if not (user and pwd and destinatario):
        return False
    msg = EmailMessage()
    msg["From"], msg["To"], msg["Subject"] = user, destinatario, asunto
    msg.set_content(cuerpo)
    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as s:
        s.login(user, pwd)
        s.send_message(msg)
    return True


In [ ]:
# @title Bot de Telegram (ejecuta esta celda y déjala corriendo)
import asyncio, nest_asyncio
from telegram import Update
from telegram.ext import (Application, CommandHandler, MessageHandler,
                          ContextTypes, filters)

nest_asyncio.apply()

# Sesiones activas por chat de Telegram:
#   chat_id -> {"thread_id", "alias", "destino", "fase": "borrador" | "bucle"}
SESIONES: dict[int, dict] = {}

# chat_id de Telegram del profesor para recibir avisos de bloqueo (opcional).
# Para obtenerlo: escríbele /start al bot y mira el id con /miid.
PROFESOR_CHAT_ID = int(os.environ.get("PROFESOR_CHAT_ID", "0")) or None

def alias_de(chat_id) -> str | None:
    for a, c in cargar_chats().items():
        if c == chat_id:
            return a
    return None

AYUDA = (
    "👋 Soy el mediador de retroalimentación del curso.\n\n"
    "1️⃣  /soy <alias>  — identifícate (alias del directorio).\n"
    "2️⃣  /retro <alias_destino>  — inicia una retroalimentación para un compañero.\n"
    "3️⃣  Pega tu comentario. Te haré preguntas y sugerencias; responde con una nueva "
    "versión o escribe ENVIAR cuando estés conforme.\n\n"
    "Otros: /cancelar, /tarea (ver la tarea actual), /tarea <texto> (cambiarla)."
)

async def cmd_start(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(AYUDA)

async def cmd_soy(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    alias = " ".join(ctx.args).strip().lower()
    if alias not in DIRECTORIO:
        return await update.message.reply_text("No encuentro ese alias en el directorio. Revisa con tu profesor.")
    guardar_chat(alias, update.effective_chat.id)
    await update.message.reply_text(f"Listo, {DIRECTORIO[alias]['nombre']}. Usa /retro <alias> para comenzar.")

async def cmd_tarea(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    global TAREA
    if ctx.args:
        TAREA = " ".join(ctx.args)
        return await update.message.reply_text("Tarea actualizada.")
    await update.message.reply_text("📋 Tarea actual:\n" + TAREA)

async def cmd_retro(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    chat_id = update.effective_chat.id
    alias = alias_de(chat_id)
    if not alias:
        return await update.message.reply_text("Primero identifícate con /soy <alias>.")
    destino = " ".join(ctx.args).strip().lower()
    if destino not in DIRECTORIO:
        return await update.message.reply_text("Ese alias de destino no existe en el directorio.")
    if destino == alias:
        return await update.message.reply_text("No puedes retroalimentarte a ti mismo 🙂")
    SESIONES[chat_id] = {"thread_id": f"tg-{chat_id}-{uuid.uuid4().hex[:8]}",
                         "alias": alias, "destino": destino, "fase": "borrador"}
    await update.message.reply_text(
        f"Retroalimentación para {DIRECTORIO[destino]['nombre']}.\n"
        f"📋 Tarea: {TAREA}\n\nPega tu comentario en un solo mensaje.")

async def cmd_miid(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(f"Tu chat_id es {update.effective_chat.id}")

async def cmd_cancelar(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    SESIONES.pop(update.effective_chat.id, None)
    await update.message.reply_text("Sesión cancelada.")

async def entregar_a_destino(ctx, destino, mensaje):
    """Entrega final: Telegram si el destinatario se registró, y correo si está configurado."""
    canales = []
    chat_destino = cargar_chats().get(destino)
    if chat_destino:
        await ctx.bot.send_message(chat_destino, "📨 " + mensaje)
        canales.append("Telegram")
    if enviar_correo(DIRECTORIO[destino].get("correo"), "Retroalimentación de un compañero", mensaje):
        canales.append("correo")
    return canales

async def avisar_profesor(ctx, sesion, texto_bloqueado):
    """Si el profesor registró su chat (PROFESOR_CHAT_ID), recibe un aviso discreto de cada bloqueo."""
    if not PROFESOR_CHAT_ID:
        return
    await ctx.bot.send_message(
        PROFESOR_CHAT_ID,
        f"⚠️ Comentario bloqueado por contenido inapropiado.\n"
        f"De: {sesion['alias']} → Para: {sesion['destino']}\nSesión: {sesion['thread_id']}\n"
        f"El texto completo está en el registro Excel.")

async def texto(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    chat_id = update.effective_chat.id
    s = SESIONES.get(chat_id)
    if not s:
        return await update.message.reply_text("Usa /retro <alias> para comenzar una retroalimentación.")
    await update.message.chat.send_action("typing")
    if s["fase"] == "borrador":
        terminado, salida, bloqueado = await asyncio.to_thread(
            iniciar_sesion, s["thread_id"], s["alias"], s["destino"], update.message.text, TAREA)
        s["fase"] = "bucle"
    else:
        terminado, salida, bloqueado = await asyncio.to_thread(
            continuar_sesion, s["thread_id"], update.message.text)
    if bloqueado:
        await avisar_profesor(ctx, s, update.message.text)
    if terminado:
        canales = await entregar_a_destino(ctx, s["destino"], salida)
        SESIONES.pop(chat_id, None)
        via = " y ".join(canales) if canales else "el registro (el destinatario aún no se ha registrado)"
        await update.message.reply_text(f"✅ Enviado a {DIRECTORIO[s['destino']]['nombre']} vía {via}. ¡Gracias!")
    else:
        await update.message.reply_text(salida)

app = Application.builder().token(os.environ["TELEGRAM_BOT_TOKEN"]).build()
for nombre, fn in [("start", cmd_start), ("ayuda", cmd_start), ("soy", cmd_soy),
                   ("retro", cmd_retro), ("cancelar", cmd_cancelar), ("tarea", cmd_tarea),
                   ("miid", cmd_miid)]:
    app.add_handler(CommandHandler(nombre, fn))
app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, texto))

print("🤖 Bot corriendo. Detén la celda para apagarlo.")
app.run_polling(stop_signals=None)


## 9. Reporte para el profesor
Resumen del Excel: evolución de puntajes por ronda, cambio inicial→final por sesión, y qué hicieron los estudiantes con las sugerencias. Descarga el archivo al final.

In [ ]:
# @title Reporte para el profesor
import pandas as pd
pd.set_option("display.max_colwidth", 80)

df = pd.read_excel(ARCHIVO_XLSX, sheet_name="iteraciones")
criterios = ["especificidad", "tono", "accionabilidad", "equilibrio", "alineacion"]
evaluadas = df.dropna(subset=["puntaje_global"])

print(f"Sesiones: {df.thread_id.nunique()}  |  Borradores evaluados: {len(evaluadas)}")
print(f"Rondas promedio por sesión: {evaluadas.groupby('thread_id').iteracion.max().mean():.2f}\n")

print("Puntaje promedio por ronda (¿mejora el feedback tras la mediación?):")
print(evaluadas.groupby("iteracion")[criterios + ["puntaje_global"]].mean().round(2), "\n")

print("Cambio de puntaje entre primera y última versión evaluada, por sesión:")
primera = evaluadas.sort_values("iteracion").groupby("thread_id").first()
ultima  = evaluadas.sort_values("iteracion").groupby("thread_id").last()
delta = (ultima.puntaje_global - primera.puntaje_global).rename("delta_puntaje")
print(pd.concat([primera[["retroalimentador", "destino"]], primera.puntaje_global.rename("inicial"),
                 ultima.puntaje_global.rename("final"), delta], axis=1).round(2), "\n")

print("Qué hicieron los estudiantes al recibir la sugerencia de la IA:")
acciones = df.accion_retroalimentador.dropna().str.split(" → ").explode()
print(acciones.value_counts(), "\n")

bloqueos = df[df.bloqueado == True]
print(f"🚫 Comentarios bloqueados por contenido inapropiado: {len(bloqueos)} "
      f"(intentos de envío bloqueados: {(acciones == 'INTENTO_ENVIO_BLOQUEADO').sum()})")
if len(bloqueos):
    print(bloqueos[["fecha_hora", "retroalimentador", "destino", "iteracion", "motivo_bloqueo"]].to_string(index=False))
print()

print("Promedio por retroalimentador (primera versión):")
print(primera.groupby("retroalimentador")[criterios].mean().round(2))

# Descargar el Excel desde Colab:
try:
    from google.colab import files
    files.download(str(ARCHIVO_XLSX))
except ImportError:
    pass
